# Trading Data → DuckDB Integration

这个 notebook 做 4 件事：
1. 扫描 `trading_data/*.csv` 的字段（attribute）与 schema 差异
2. 导出字段统计报告
3. 将全部文件写入 `trading_data.duckdb`
4. 提供基础校验查询

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import duckdb

ROOT = Path('.')
DATA_DIR = ROOT / 'trading_data'
DB_PATH = ROOT / 'trading_data.duckdb'
ATTR_REPORT = ROOT / 'trading_data_attribute_frequency.csv'
SCHEMA_REPORT = ROOT / 'trading_data_schema_frequency.csv'

ENCODINGS = ['utf-8-sig', 'gb18030', 'gbk']
files = sorted(DATA_DIR.glob('*.csv'))
print(f'CSV file count: {len(files)}')

CSV file count: 5739


In [2]:
def read_header_with_encoding(path: Path):
    last_err = None
    for enc in ENCODINGS:
        try:
            cols = list(pd.read_csv(path, nrows=0, encoding=enc).columns)
            cols = [str(c).strip() for c in cols]
            return cols, enc
        except Exception as e:
            last_err = e
    raise RuntimeError(f'Cannot read header: {path.name} | {last_err}')

header_map = {}
encoding_map = {}
attribute_counter = Counter()
schema_counter = Counter()

for i, f in enumerate(files, 1):
    cols, enc = read_header_with_encoding(f)
    header_map[f] = cols
    encoding_map[f] = enc
    attribute_counter.update(cols)
    schema_counter[tuple(cols)] += 1
    if i % 500 == 0:
        print(f'Header scanned: {i}/{len(files)}')

all_attributes = sorted(attribute_counter.keys())
print('Unique attributes:', len(all_attributes))
print('Unique schemas:', len(schema_counter))

Header scanned: 500/5739
Header scanned: 1000/5739
Header scanned: 1500/5739
Header scanned: 2000/5739
Header scanned: 2500/5739
Header scanned: 3000/5739
Header scanned: 3500/5739
Header scanned: 4000/5739
Header scanned: 4500/5739
Header scanned: 5000/5739
Header scanned: 5500/5739
Unique attributes: 41
Unique schemas: 5


In [3]:
attr_df = pd.DataFrame({
    'attribute': list(attribute_counter.keys()),
    'file_count': list(attribute_counter.values())
}).sort_values(['file_count', 'attribute'], ascending=[False, True]).reset_index(drop=True)

schema_rows = []
for schema, cnt in schema_counter.items():
    schema_rows.append({
        'file_count': cnt,
        'column_count': len(schema),
        'columns_joined': ' | '.join(schema)
    })
schema_df = pd.DataFrame(schema_rows).sort_values('file_count', ascending=False).reset_index(drop=True)

attr_df.to_csv(ATTR_REPORT, index=False, encoding='utf-8-sig')
schema_df.to_csv(SCHEMA_REPORT, index=False, encoding='utf-8-sig')

display(attr_df.head(30))
display(schema_df.head(10))
print(f'Saved: {ATTR_REPORT}')
print(f'Saved: {SCHEMA_REPORT}')

,attribute,file_count
0,交易日期,5739
1,前收盘价,5739
2,开盘价,5739
3,成交量(手),5739
4,收盘价,5739
5,最低价,5739
6,最高价,5739
7,涨跌幅(%),5739
8,涨跌额,5739
9,股票代码,5739


,file_count,column_count,columns_joined
0,5124,34,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...
1,607,33,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...
2,6,21,股票代码 | 交易日期 | 开盘价 | 最高价 | 最低价 | 收盘价 | 前收盘价 | 涨...
3,1,31,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...
4,1,18,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...


Saved: trading_data_attribute_frequency.csv
Saved: trading_data_schema_frequency.csv


In [10]:
# Ensure previous connection is closed before recreating DB
try:
    conn.close()
except Exception:
    pass

if DB_PATH.exists():
    DB_PATH.unlink()

db_columns = all_attributes + ['source_file']

def q(col: str) -> str:
    return '"' + col.replace('"', '""') + '"'

create_cols_sql = ', '.join([f"{q(c)} VARCHAR" for c in db_columns])
create_sql = f"CREATE TABLE trading_data_raw ({create_cols_sql});"

conn = duckdb.connect(str(DB_PATH))
conn.execute(create_sql)
print(f'Created DB: {DB_PATH}')
print(f'Raw column count: {len(db_columns)}')

Created DB: trading_data.duckdb
Raw column count: 42


In [11]:
inserted_rows = 0
for i, f in enumerate(files, 1):
    enc = encoding_map[f]
    df = pd.read_csv(f, encoding=enc)
    df.columns = [str(c).strip() for c in df.columns]

    missing_cols = [c for c in all_attributes if c not in df.columns]
    for c in missing_cols:
        df[c] = None

    ordered = df[all_attributes].copy()
    ordered['source_file'] = f.name

    conn.register('tmp_df', ordered)
    conn.execute('INSERT INTO trading_data_raw SELECT * FROM tmp_df')
    conn.unregister('tmp_df')

    inserted_rows += len(ordered)
    if i % 200 == 0 or i == len(files):
        print(f'Loaded files: {i}/{len(files)} | rows inserted: {inserted_rows:,}')

print(f'Total inserted rows: {inserted_rows:,}')

Loaded files: 200/5739 | rows inserted: 1,172,831
Loaded files: 400/5739 | rows inserted: 2,359,886
Loaded files: 600/5739 | rows inserted: 2,917,416
Loaded files: 800/5739 | rows inserted: 3,763,051
Loaded files: 1000/5739 | rows inserted: 4,497,919
Loaded files: 1200/5739 | rows inserted: 5,157,544
Loaded files: 1400/5739 | rows inserted: 5,598,012
Loaded files: 1600/5739 | rows inserted: 6,114,883
Loaded files: 1800/5739 | rows inserted: 6,789,289
Loaded files: 2000/5739 | rows inserted: 7,304,576
Loaded files: 2200/5739 | rows inserted: 7,714,510
Loaded files: 2400/5739 | rows inserted: 7,981,047
Loaded files: 2600/5739 | rows inserted: 8,180,967
Loaded files: 2800/5739 | rows inserted: 8,326,800
Loaded files: 3000/5739 | rows inserted: 8,931,448
Loaded files: 3200/5739 | rows inserted: 10,107,719
Loaded files: 3400/5739 | rows inserted: 11,217,660
Loaded files: 3600/5739 | rows inserted: 12,419,451
Loaded files: 3800/5739 | rows inserted: 13,056,093
Loaded files: 4000/5739 | rows 

In [15]:
try:
    conn.close()
except Exception:
    pass
conn = duckdb.connect(str(DB_PATH))

def first_existing(candidates):
    for c in candidates:
        if c in all_attributes:
            return c
    return None


def text_expr(candidates, alias):
    col = first_existing(candidates)
    if col is None:
        return f"NULL::VARCHAR AS {alias}"
    return f'TRIM("{col}")::VARCHAR AS {alias}'


def date_expr(candidates, alias):
    col = first_existing(candidates)
    if col is None:
        return f"NULL::DATE AS {alias}"
    return f"""
    CASE
        WHEN REGEXP_MATCHES(TRIM(\"{col}\"), '^[0-9]{{8}}(?:\\.0+)?$') THEN
            STRPTIME(SUBSTR(TRIM(\"{col}\"), 1, 8), '%Y%m%d')::DATE
        WHEN REGEXP_MATCHES(TRIM(\"{col}\"), '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$') THEN
            STRPTIME(TRIM(\"{col}\"), '%Y-%m-%d')::DATE
        WHEN REGEXP_MATCHES(TRIM(\"{col}\"), '^[0-9]{{4}}/[0-9]{{2}}/[0-9]{{2}}$') THEN
            STRPTIME(TRIM(\"{col}\"), '%Y/%m/%d')::DATE
        ELSE NULL
    END AS {alias}
    """.strip()


def numeric_expr(candidates, alias, multiplier=1.0):
    col = first_existing(candidates)
    if col is None:
        return f"NULL::DOUBLE AS {alias}"
    cleaned = f"REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(TRIM(\"{col}\"), ',', ''), '，', ''), '%', '')"
    if multiplier == 1.0:
        return f"TRY_CAST(NULLIF({cleaned}, '') AS DOUBLE) AS {alias}"
    return f"(TRY_CAST(NULLIF({cleaned}, '') AS DOUBLE) * {multiplier}) AS {alias}"


turnover_amount_expr = numeric_expr(['成交金额(元)', '成交额(元)', '成交额(千元)', '成交额(万元)'], 'turnover_amount')
if first_existing(['成交额(千元)']) is not None and first_existing(['成交金额(元)', '成交额(元)']) is None:
    turnover_amount_expr = numeric_expr(['成交额(千元)'], 'turnover_amount', 1000.0)
elif first_existing(['成交额(万元)']) is not None and first_existing(['成交金额(元)', '成交额(元)', '成交额(千元)']) is None:
    turnover_amount_expr = numeric_expr(['成交额(万元)'], 'turnover_amount', 10000.0)

free_float_expr = numeric_expr(['流通市值(元)', '流通市值(万元)'], 'free_float_mkt_cap')
if first_existing(['流通市值(万元)']) is not None and first_existing(['流通市值(元)']) is None:
    free_float_expr = numeric_expr(['流通市值(万元)'], 'free_float_mkt_cap', 10000.0)

total_mkt_cap_expr = numeric_expr(['总市值(元)', '总市值(万元)'], 'total_mkt_cap')
if first_existing(['总市值(万元)']) is not None and first_existing(['总市值(元)']) is None:
    total_mkt_cap_expr = numeric_expr(['总市值(万元)'], 'total_mkt_cap', 10000.0)

create_typed_sql = """
DROP TABLE IF EXISTS trading_data_en;
CREATE TABLE trading_data_en (
    source_file VARCHAR,
    stock_code VARCHAR,
    stock_name VARCHAR,
    industry VARCHAR,
    region VARCHAR,
    ts_code VARCHAR,
    list_date DATE,
    trade_date DATE,
    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE,
    prev_close DOUBLE,
    change_amt DOUBLE,
    change_pct DOUBLE,
    volume_lot DOUBLE,
    turnover_amount DOUBLE,
    turnover_pct DOUBLE,
    pe DOUBLE,
    pb DOUBLE,
    ps DOUBLE,
    dividend_yield_pct DOUBLE,
    free_float_mkt_cap DOUBLE,
    total_mkt_cap DOUBLE
);
"""
conn.execute(create_typed_sql)

insert_typed_sql = f"""
INSERT INTO trading_data_en
SELECT
    source_file::VARCHAR AS source_file,
    {text_expr(['股票代码'], 'stock_code')},
    {text_expr(['名称'], 'stock_name')},
    {text_expr(['所属行业'], 'industry')},
    {text_expr(['地域'], 'region')},
    {text_expr(['TS代码'], 'ts_code')},
    {date_expr(['上市日期'], 'list_date')},
    {date_expr(['交易日期'], 'trade_date')},
    {numeric_expr(['开盘价'], 'open')},
    {numeric_expr(['最高价'], 'high')},
    {numeric_expr(['最低价'], 'low')},
    {numeric_expr(['收盘价'], 'close')},
    {numeric_expr(['前收盘价'], 'prev_close')},
    {numeric_expr(['涨跌额'], 'change_amt')},
    {numeric_expr(['涨跌幅(%)'], 'change_pct')},
    {numeric_expr(['成交量(手)'], 'volume_lot')},
    {turnover_amount_expr},
    {numeric_expr(['换手率(%)'], 'turnover_pct')},
    {numeric_expr(['市盈率', '市盈率(TTM,亏损的PE为空)'], 'pe')},
    {numeric_expr(['市净率'], 'pb')},
    {numeric_expr(['市销率', '市销率(TTM)'], 'ps')},
    {numeric_expr(['股息率(%)'], 'dividend_yield_pct')},
    {free_float_expr},
    {total_mkt_cap_expr}
FROM trading_data_raw;
"""
conn.execute(insert_typed_sql)

conn.execute("CREATE OR REPLACE VIEW trading_data_clean AS SELECT * FROM trading_data_en")
print('Created table: trading_data_en (English columns with explicit schema)')
print('Created view: trading_data_clean -> trading_data_en')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created table: trading_data_en (English columns with explicit schema)
Created view: trading_data_clean -> trading_data_en


In [16]:
summary_df = conn.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT stock_code) AS stock_count,
    MIN(trade_date) AS min_trade_date,
    MAX(trade_date) AS max_trade_date
FROM trading_data_en
""").fetchdf()
display(summary_df)

null_check_df = conn.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trade_date IS NULL THEN 1 ELSE 0 END) AS null_trade_date_rows,
    SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close_rows,
    SUM(CASE WHEN total_mkt_cap IS NULL THEN 1 ELSE 0 END) AS null_total_mkt_cap_rows
FROM trading_data_en
""").fetchdf()
display(null_check_df)

schema_df = conn.execute("""
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'trading_data_en'
ORDER BY ordinal_position
""").fetchdf()
display(schema_df)

sample_df = conn.execute("""
SELECT *
FROM trading_data_en
ORDER BY trade_date DESC NULLS LAST
LIMIT 20
""").fetchdf()
display(sample_df)

,row_count,stock_count,min_trade_date,max_trade_date
0,15800649,5739,1993-01-07,2025-12-05


,total_rows,null_trade_date_rows,null_close_rows,null_total_mkt_cap_rows
0,15800649,11326.0,11326.0,396011.0


,table_name,column_name,data_type
0,trading_data_en,source_file,VARCHAR
1,trading_data_en,stock_code,VARCHAR
2,trading_data_en,stock_name,VARCHAR
3,trading_data_en,industry,VARCHAR
4,trading_data_en,region,VARCHAR
5,trading_data_en,ts_code,VARCHAR
6,trading_data_en,list_date,DATE
7,trading_data_en,trade_date,DATE
8,trading_data_en,open,DOUBLE
9,trading_data_en,high,DOUBLE


,source_file,stock_code,stock_name,industry,region,ts_code,list_date,trade_date,open,high,...,change_pct,volume_lot,turnover_amount,turnover_pct,pe,pb,ps,dividend_yield_pct,free_float_mkt_cap,total_mkt_cap
0,000548.csv,548,湖南投资,路桥,湖南,000548.SZ,1993-12-20,2025-12-05,5.85,5.92,...,0.8547,88208.00,5.165725e+07,1.7671,36.1736,1.4243,4.6664,1.5254,2.945127e+09,2.945373e+09
1,000550.csv,550,江铃汽车,汽车整车,江西,000550.SZ,1993-12-01,2025-12-05,18.71,18.78,...,0.0534,20986.00,3.913202e+07,0.4048,10.5182,1.4363,0.4213,3.6519,9.710815e+09,1.616800e+10
2,000551.csv,551,创元科技,专用机械,江苏,000551.SZ,1994-01-06,2025-12-05,10.99,11.17,...,0.9050,65780.88,7.293775e+07,1.3620,21.5648,1.9208,1.2893,0.5232,5.385253e+09,5.404773e+09
3,000552.csv,552,甘肃能化,煤炭开采,甘肃,000552.SZ,1994-01-06,2025-12-05,2.47,2.48,...,0.4049,396741.19,9.781640e+07,1.0648,10.9300,0.8120,1.3820,4.0322,9.239999e+09,1.327249e+10
4,000553.csv,553,安道麦A,农药化肥,湖北,000553.SZ,1993-12-03,2025-12-05,6.03,6.20,...,1.9835,42491.26,2.601252e+07,0.1952,NaN,0.7790,0.4875,0.0000,1.343253e+10,1.437494e+10
5,000554.csv,554,泰山石油,石油贸易,山东,000554.SZ,1993-12-15,2025-12-05,6.72,6.77,...,0.5952,73937.95,4.974989e+07,2.0388,32.7210,2.8799,0.9832,0.6361,2.451536e+09,3.250163e+09
6,000555.csv,555,神州信息,软件服务,深圳,000555.SZ,1994-04-08,2025-12-05,16.60,17.11,...,2.5904,411897.94,6.964731e+08,4.2381,NaN,2.9987,1.6613,0.1854,1.655152e+10,1.661744e+10
7,000557.csv,557,西部创业,铁路,宁夏,000557.SZ,1994-06-17,2025-12-05,4.95,5.00,...,1.0101,58300.81,2.891955e+07,0.3999,27.7249,1.1453,5.4202,0.0000,7.289581e+09,7.291874e+09
8,000558.csv,558,莱茵体育,旅游景点,四川,000558.SZ,1994-05-09,2025-12-05,5.59,5.66,...,-2.8319,843495.30,4.670464e+08,6.5454,NaN,6.7489,17.1407,0.0000,7.074878e+09,7.077839e+09
9,000559.csv,559,万向钱潮,汽车配件,浙江,000559.SZ,1994-01-10,2025-12-05,12.61,12.72,...,-2.8235,1370748.74,1.691018e+09,4.1356,43.1937,4.3700,3.1921,1.2064,4.106720e+10,4.107729e+10


In [17]:
# Missing diagnostics: find where nulls come from
diag_null_trade = conn.execute("""
SELECT source_file, COUNT(*) AS null_trade_date_rows
FROM trading_data_en
WHERE trade_date IS NULL
GROUP BY source_file
ORDER BY null_trade_date_rows DESC
LIMIT 30
""").fetchdf()
display(diag_null_trade)

diag_raw_date_pattern = conn.execute("""
SELECT
    REGEXP_REPLACE(TRIM("交易日期"), '[0-9]', '9', 'g') AS date_pattern,
    COUNT(*) AS rows
FROM trading_data_raw
WHERE "交易日期" IS NOT NULL AND TRIM("交易日期") <> ''
GROUP BY 1
ORDER BY rows DESC
LIMIT 30
""").fetchdf()
display(diag_raw_date_pattern)

diag_problematic_examples = conn.execute("""
SELECT
    source_file,
    "交易日期" AS raw_trade_date,
    "收盘价" AS raw_close,
    "前收盘价" AS raw_prev_close,
    "涨跌幅(%)" AS raw_change_pct
FROM trading_data_raw
WHERE "交易日期" IS NOT NULL
  AND TRIM("交易日期") <> ''
  AND NOT (
      LENGTH(TRIM("交易日期")) = 8
      OR TRIM("交易日期") LIKE '____-__-__'
  )
LIMIT 50
""").fetchdf()
display(diag_problematic_examples)

diag_missing_key_fields = conn.execute("""
SELECT
    SUM(CASE WHEN trade_date IS NULL THEN 1 ELSE 0 END) AS null_trade_date_rows,
    SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close_rows,
    SUM(CASE WHEN prev_close IS NULL THEN 1 ELSE 0 END) AS null_prev_close_rows,
    SUM(CASE WHEN change_pct IS NULL THEN 1 ELSE 0 END) AS null_change_pct_rows
FROM trading_data_en
""").fetchdf()
display(diag_missing_key_fields)

,source_file,null_trade_date_rows
0,002569.csv,541
1,002289.csv,448
2,300163.csv,419
3,002571.csv,408
4,002721.csv,404
5,300131.csv,384
6,300353.csv,341
7,002633.csv,315
8,300423.csv,268
9,002515.csv,256


,date_pattern,rows
0,99999999,15378552
1,99999999.9,410771


,source_file,raw_trade_date,raw_close,raw_prev_close,raw_change_pct
0,001206.csv,20251205.0,30.23,30.53,-0.9826
1,001206.csv,20251204.0,30.53,31.3,-2.4601
2,001206.csv,20251203.0,31.3,31.74,-1.3863
3,001206.csv,20251202.0,31.74,31.95,-0.6573
4,001206.csv,20251201.0,31.95,30.59,4.4459
5,001206.csv,20251128.0,30.59,30.46,0.4268
6,001206.csv,20251127.0,30.46,30.9,-1.4239
7,001206.csv,20251126.0,30.9,30.9,0.0
8,001206.csv,20251125.0,30.9,31.2,-0.9615
9,001206.csv,20251124.0,31.2,33.13,-5.8255


,null_trade_date_rows,null_close_rows,null_prev_close_rows,null_change_pct_rows
0,11326.0,11326.0,11802.0,11802.0


In [23]:
# Compare stock_code: 00001 vs 000001.SH
from IPython.display import display

codes_to_check = ['1', '000001.SH']

for code in codes_to_check:
    print(f"\n=== stock_code = {code} ===")
    summary = conn.execute("""
        SELECT
            COUNT(*) AS rows,
            MIN(trade_date) AS min_trade_date,
            MAX(trade_date) AS max_trade_date,
            SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close_rows,
            SUM(CASE WHEN change_pct IS NULL THEN 1 ELSE 0 END) AS null_change_pct_rows
        FROM trading_data_en
        WHERE stock_code = ?
    """, [code]).fetchdf()
    display(summary)

    latest_rows = conn.execute("""
        SELECT
            trade_date, stock_code, stock_name, close, prev_close, change_pct, source_file
        FROM trading_data_en
        WHERE stock_code = ?
        ORDER BY trade_date DESC NULLS LAST
        LIMIT 20
    """, [code]).fetchdf()
    display(latest_rows)


=== stock_code = 1 ===


,rows,min_trade_date,max_trade_date,null_close_rows,null_change_pct_rows
0,6122,2000-01-04,2025-12-05,0.0,0.0


,trade_date,stock_code,stock_name,close,prev_close,change_pct,source_file
0,2025-12-05,1,平安银行,11.53,11.49,0.3481,000001.csv
1,2025-12-04,1,平安银行,11.49,11.55,-0.5195,000001.csv
2,2025-12-03,1,平安银行,11.55,11.64,-0.7732,000001.csv
3,2025-12-02,1,平安银行,11.64,11.69,-0.4277,000001.csv
4,2025-12-01,1,平安银行,11.69,11.61,0.6891,000001.csv
5,2025-11-28,1,平安银行,11.61,11.71,-0.8540,000001.csv
6,2025-11-27,1,平安银行,11.71,11.69,0.1711,000001.csv
7,2025-11-26,1,平安银行,11.69,11.80,-0.9322,000001.csv
8,2025-11-25,1,平安银行,11.80,11.60,1.7241,000001.csv
9,2025-11-24,1,平安银行,11.60,11.69,-0.7699,000001.csv



=== stock_code = 000001.SH ===


,rows,min_trade_date,max_trade_date,null_close_rows,null_change_pct_rows
0,8000,1993-02-01,2025-12-05,0.0,0.0


,trade_date,stock_code,stock_name,close,prev_close,change_pct,source_file
0,2025-12-05,000001.SH,None,3902.8076,3875.7933,0.70,上证指数数据.csv
1,2025-12-04,000001.SH,None,3875.7933,3877.9996,-0.06,上证指数数据.csv
2,2025-12-03,000001.SH,None,3877.9996,3897.7116,-0.51,上证指数数据.csv
3,2025-12-02,000001.SH,None,3897.7116,3914.0061,-0.42,上证指数数据.csv
4,2025-12-01,000001.SH,None,3914.0061,3888.5962,0.65,上证指数数据.csv
5,2025-11-28,000001.SH,None,3888.5962,3875.2594,0.34,上证指数数据.csv
6,2025-11-27,000001.SH,None,3875.2594,3864.1844,0.29,上证指数数据.csv
7,2025-11-26,000001.SH,None,3864.1844,3870.0228,-0.15,上证指数数据.csv
8,2025-11-25,000001.SH,None,3870.0228,3836.7655,0.87,上证指数数据.csv
9,2025-11-24,000001.SH,None,3836.7655,3834.8908,0.05,上证指数数据.csv


In [8]:
conn.close()
print('DuckDB connection closed.')

DuckDB connection closed.
